# Evaluation Analysis — SFT vs GRPO

Analyses `data/evaluation_results.csv` (500 test samples).

Columns: `expected`, `sft_predicted`, `sft_reward`, `grpo_predicted`, `grpo_reward`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('../data/evaluation_results.csv')
df['expected_action'] = df['expected'].str.split().str[0]
df['sft_action']      = df['sft_predicted'].str.split().str[0]
df['grpo_action']     = df['grpo_predicted'].str.split().str[0]

print(f'Rows: {len(df):,}')
df.head()

## 1. Overall accuracy

In [ ]:
sft_acc  = (df['sft_reward']  == 1.0).mean() * 100
grpo_acc = (df['grpo_reward'] == 1.0).mean() * 100

sft_avg  = df['sft_reward'].mean()
grpo_avg = df['grpo_reward'].mean()

print(f'SFT  — exact accuracy: {sft_acc:.1f}%  |  avg reward: {sft_avg:.3f}')
print(f'GRPO — exact accuracy: {grpo_acc:.1f}%  |  avg reward: {grpo_avg:.3f}')
print(f'\nGRPO improvement: {grpo_acc - sft_acc:+.1f}% accuracy  |  {grpo_avg - sft_avg:+.3f} avg reward')

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(['SFT', 'GRPO'], [sft_acc, grpo_acc], color=['steelblue', 'coral'], width=0.4)
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_ylim(0, 110)
ax.set_ylabel('Exact accuracy (%)')
ax.set_title('SFT vs GRPO — overall accuracy')
plt.tight_layout()
plt.show()

## 2. Reward distribution

In [ ]:
reward_vals = sorted(df['sft_reward'].unique())

sft_counts  = df['sft_reward'].value_counts().reindex(reward_vals, fill_value=0)
grpo_counts = df['grpo_reward'].value_counts().reindex(reward_vals, fill_value=0)

x = np.arange(len(reward_vals))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, sft_counts,  w, label='SFT',  color='steelblue')
ax.bar(x + w/2, grpo_counts, w, label='GRPO', color='coral')
ax.set_xticks(x)
ax.set_xticklabels([str(v) for v in reward_vals])
ax.set_xlabel('Reward')
ax.set_ylabel('Count')
ax.set_title('Reward distribution — SFT vs GRPO')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Accuracy by action type

In [ ]:
actions = ['check', 'call', 'fold', 'bet', 'raise']

sft_by_action  = []
grpo_by_action = []
counts         = []

for action in actions:
    mask = df['expected_action'] == action
    n = mask.sum()
    counts.append(n)
    sft_by_action.append((df.loc[mask, 'sft_reward']  == 1.0).mean() * 100)
    grpo_by_action.append((df.loc[mask, 'grpo_reward'] == 1.0).mean() * 100)

x = np.arange(len(actions))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, sft_by_action,  w, label='SFT',  color='steelblue')
ax.bar(x + w/2, grpo_by_action, w, label='GRPO', color='coral')
ax.set_xticks(x)
ax.set_xticklabels([f'{a}\n(n={c})' for a, c in zip(actions, counts)])
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 110)
ax.set_title('Accuracy by action type')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Where did GRPO improve / regress?

In [ ]:
df['reward_delta'] = df['grpo_reward'] - df['sft_reward']

improved  = (df['reward_delta'] > 0).sum()
regressed = (df['reward_delta'] < 0).sum()
same      = (df['reward_delta'] == 0).sum()

print(f'GRPO improved : {improved} rows  ({improved/len(df)*100:.1f}%)')
print(f'GRPO regressed: {regressed} rows  ({regressed/len(df)*100:.1f}%)')
print(f'No change     : {same} rows  ({same/len(df)*100:.1f}%)')

print('\n--- Cases where GRPO improved ---')
display(df[df['reward_delta'] > 0][['expected','sft_predicted','grpo_predicted','sft_reward','grpo_reward']].head(10))

print('\n--- Cases where GRPO regressed ---')
display(df[df['reward_delta'] < 0][['expected','sft_predicted','grpo_predicted','sft_reward','grpo_reward']].head(10))

## 5. SFT confusion matrix (action types)

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, pred_col in zip(axes, ['SFT', 'GRPO'], ['sft_action', 'grpo_action']):
    cm = confusion_matrix(df['expected_action'], df[pred_col], labels=actions)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=actions, yticklabels=actions,
                cmap='Blues', ax=ax)
    ax.set_title(f'{model} — action confusion matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Expected')

plt.tight_layout()
plt.show()

## 6. Bet/raise sizing accuracy

In [ ]:
sizing = df[df['expected_action'].isin(['bet', 'raise'])].copy()

def extract_amount(s):
    try:
        return float(str(s).split()[-1])
    except:
        return None

sizing['expected_amt'] = sizing['expected'].apply(extract_amount)
sizing['sft_amt']      = sizing['sft_predicted'].apply(extract_amount)
sizing['grpo_amt']     = sizing['grpo_predicted'].apply(extract_amount)

valid_sft  = sizing.dropna(subset=['expected_amt', 'sft_amt'])
valid_grpo = sizing.dropna(subset=['expected_amt', 'grpo_amt'])

sft_ratio  = valid_sft['sft_amt']  / valid_sft['expected_amt']
grpo_ratio = valid_grpo['grpo_amt'] / valid_grpo['expected_amt']

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sft_ratio,  bins=40, alpha=0.6, label='SFT',  color='steelblue', range=(0, 3))
ax.hist(grpo_ratio, bins=40, alpha=0.6, label='GRPO', color='coral',     range=(0, 3))
ax.axvline(1.0, color='black', linestyle='--', label='Perfect sizing')
ax.set_xlabel('Predicted / Expected amount ratio')
ax.set_ylabel('Count')
ax.set_title('Bet/raise sizing ratio (1.0 = perfect)')
ax.legend()
plt.tight_layout()
plt.show()

within_10_sft  = ((sft_ratio  >= 0.9) & (sft_ratio  <= 1.1)).mean() * 100
within_10_grpo = ((grpo_ratio >= 0.9) & (grpo_ratio <= 1.1)).mean() * 100
print(f'Sizing within 10%  — SFT: {within_10_sft:.1f}%  |  GRPO: {within_10_grpo:.1f}%')